In [3]:
import pandas as pd
import numpy as np


In [ ]:
# ---------- Load data ----------
dim_date    = pd.read_csv("C:\\Users\\slkum\\Downloads\\RPC12_Input_For_Participants\\datasets\\dim_date.csv")
makers      = pd.read_csv("C:\\Users\\slkum\\Downloads\\RPC12_Input_For_Participants\\datasets\\electric_vehicle_sales_by_makers.csv")
states      = pd.read_csv("C:\\Users\\slkum\\Downloads\\RPC12_Input_For_Participants\\datasets\\electric_vehicle_sales_by_state.csv")


In [9]:
# ---------- Standardise date columns ----------
for df in [dim_date, makers, states]:
    df['date'] = pd.to_datetime(df['date'], format='%d-%b-%y')


In [ ]:
# ---------- Merge fiscal_year & quarter into fact tables ----------

dim_date_small = dim_date[['date', 'fiscal_year', 'quarter']]

makers = makers.merge(dim_date_small, on='date', how='left')
states = states.merge(dim_date_small, on='date', how='left')


In [11]:
# ---------- Sanity checks ----------
print(makers.head())
print(states.head())
print("Fiscal years:", sorted(makers.fiscal_year.unique()))


        date vehicle_category         maker  electric_vehicles_sold  \
0 2021-04-01       2-Wheelers  OLA ELECTRIC                       0   
1 2022-04-01       2-Wheelers      OKAYA EV                       0   
2 2021-05-01       2-Wheelers  OLA ELECTRIC                       0   
3 2021-06-01       2-Wheelers  OLA ELECTRIC                       0   
4 2021-07-01       2-Wheelers  OLA ELECTRIC                       0   

   fiscal_year quarter  
0         2022      Q1  
1         2023      Q1  
2         2022      Q1  
3         2022      Q1  
4         2022      Q2  
        date   state vehicle_category  electric_vehicles_sold  \
0 2021-04-01  Sikkim       2-Wheelers                       0   
1 2021-04-01  Sikkim       4-Wheelers                       0   
2 2021-05-01  Sikkim       2-Wheelers                       0   
3 2021-05-01  Sikkim       4-Wheelers                       0   
4 2021-06-01  Sikkim       2-Wheelers                       0   

   total_vehicles_sold  fiscal_y

In [ ]:
### Q1. Top 3 and Bottom 3 makers for FY 2023 and FY 2024 (2-Wheelers sold)
two_w = makers[makers.vehicle_category == '2-Wheelers']

def top_bottom(df, fy, n=3):
    d = (df[df.fiscal_year == fy]
         .groupby('maker')['electric_vehicles_sold']
         .sum()
         .sort_values(ascending=False))
    return d.head(n), d.tail(n)

for fy in [2023, 2024]:
    top, bottom = top_bottom(two_w, fy)
    print(f"\n===== FY {fy} – TOP 3 2W MAKERS =====")
    print(top)
    print(f"===== FY {fy} – BOTTOM 3 2W MAKERS =====")
    print(bottom)


===== FY 2023 – TOP 3 2W MAKERS =====
maker
OLA ELECTRIC     152583
OKINAWA           96945
HERO ELECTRIC     88993
Name: electric_vehicles_sold, dtype: int64
===== FY 2023 – BOTTOM 3 2W MAKERS =====
maker
PURE EV     11556
BEING       11018
JITENDRA     8563
Name: electric_vehicles_sold, dtype: int64

===== FY 2024 – TOP 3 2W MAKERS =====
maker
OLA ELECTRIC    322489
TVS             180743
ATHER           107552
Name: electric_vehicles_sold, dtype: int64
===== FY 2024 – BOTTOM 3 2W MAKERS =====
maker
KINETIC GREEN      9585
REVOLT             7254
BATTRE ELECTRIC    4841
Name: electric_vehicles_sold, dtype: int64


In [13]:
## Q2. Top 5 states by penetration rate (2W & 4W) in FY 2024

fy24 = states[states.fiscal_year == 2024]

pen = (fy24.groupby(['state', 'vehicle_category'])
       .agg(ev=('electric_vehicles_sold', 'sum'),
            total=('total_vehicles_sold', 'sum'))
       .reset_index())
pen['penetration_%'] = (pen.ev / pen.total) * 100

for cat in ['2-Wheelers', '4-Wheelers']:
    top5 = (pen[pen.vehicle_category == cat]
            .sort_values('penetration_%', ascending=False)
            .head(5))
    print(f"\nTop 5 states – {cat} EV penetration FY2024")
    print(top5[['state', 'penetration_%']])



Top 5 states – 2-Wheelers EV penetration FY2024
          state  penetration_%
20          Goa      17.992264
34       Kerala      13.524903
32    Karnataka      11.573279
40  Maharashtra      10.072507
18        Delhi       9.400866

Top 5 states – 4-Wheelers EV penetration FY2024
         state  penetration_%
35      Kerala       5.758445
13  Chandigarh       4.503112
19       Delhi       4.290757
33   Karnataka       4.261120
21         Goa       4.254353


In [14]:
## Q3. States with negative penetration / decline from 2022 → 2024

piv = (states.groupby(['state', 'fiscal_year'])
       .agg(ev=('electric_vehicles_sold', 'sum'),
            total=('total_vehicles_sold', 'sum'))
       .reset_index())
piv['pen_%'] = (piv.ev / piv.total) * 100

wide = piv.pivot(index='state', columns='fiscal_year', values='pen_%')
decline = wide[(wide[2024] < wide[2022])]
print("States with decline in penetration 2022 → 2024:")
print(decline[[2022, 2023, 2024]])



States with decline in penetration 2022 → 2024:
Empty DataFrame
Columns: [2022, 2023, 2024]
Index: []


In [15]:
## Q4. Quarterly trends – Top 5 EV makers (4-Wheelers), 2022-2024
four_w = makers[(makers.vehicle_category == '4-Wheelers') &
                (makers.fiscal_year.between(2022, 2024))]

top5_makers = (four_w.groupby('maker')['electric_vehicles_sold']
               .sum().nlargest(5).index.tolist())

trend = (four_w[four_w.maker.isin(top5_makers)]
         .groupby(['maker', 'fiscal_year', 'quarter'])
         ['electric_vehicles_sold'].sum().reset_index())

print(trend.pivot_table(index=['fiscal_year','quarter'],
                        columns='maker',
                        values='electric_vehicles_sold'))

maker                BYD India  Hyundai Motor  MG Motor  Mahindra & Mahindra  \
fiscal_year quarter                                                            
2022        Q1             0.0           25.0     285.0                355.0   
            Q2             0.0           34.0     798.0                651.0   
            Q3             1.0           25.0     411.0               1383.0   
            Q4            32.0           26.0     153.0               1653.0   
2023        Q1            81.0           75.0     531.0               2020.0   
            Q2           113.0          155.0     635.0               3164.0   
            Q3           103.0          191.0    1165.0               3378.0   
            Q4           623.0          155.0     946.0               5243.0   
2024        Q1           406.0          292.0    1493.0              10911.0   
            Q2           310.0          390.0    2524.0               5855.0   
            Q3           350.0          

In [16]:
## Q5. Delhi vs Karnataka – EV Sales & Penetration (2024)

comp = (states[(states.fiscal_year == 2024) &
               (states.state.isin(['Delhi','Karnataka']))]
        .groupby(['state','vehicle_category'])
        .agg(ev=('electric_vehicles_sold','sum'),
             total=('total_vehicles_sold','sum'))
        .reset_index())
comp['pen_%'] = comp.ev / comp.total * 100
print(comp)

       state vehicle_category      ev    total      pen_%
0      Delhi       2-Wheelers   38094   405218   9.400866
1      Delhi       4-Wheelers    8630   201130   4.290757
2  Karnataka       2-Wheelers  148111  1279767  11.573279
3  Karnataka       4-Wheelers   12878   302221   4.261120


In [17]:
## Q6. CAGR of 4-Wheeler units – Top 5 makers (2022 → 2024)

four = makers[(makers.vehicle_category=='4-Wheelers') &
              (makers.fiscal_year.isin([2022,2024]))]

piv = (four.groupby(['maker','fiscal_year'])['electric_vehicles_sold']
       .sum().unstack())

top5 = (makers[(makers.vehicle_category=='4-Wheelers')]
        .groupby('maker')['electric_vehicles_sold'].sum()
        .nlargest(5).index)

piv = piv.loc[top5]
piv['CAGR_%'] = ((piv[2024]/piv[2022]) ** (1/2) - 1) * 100
print(piv)

fiscal_year           2022   2024      CAGR_%
maker                                        
Tata Motors          12708  48181   94.714952
Mahindra & Mahindra   4042  23346  140.330055
MG Motor              1647   8829  131.530899
BYD India               33   1466  566.515134
Hyundai Motor          110   1390  255.476633


In [18]:
## Q7. Top 10 states by CAGR (Total vehicles sold, 2022 → 2024)

tot = (states.groupby(['state','fiscal_year'])['total_vehicles_sold']
       .sum().unstack())

tot = tot.dropna(subset=[2022, 2024])
tot['CAGR_%'] = ((tot[2024]/tot[2022]) ** (1/2) - 1) * 100
print(tot.sort_values('CAGR_%', ascending=False).head(10))

fiscal_year             2022       2023       2024     CAGR_%
state                                                        
Meghalaya            22193.0    31362.0    36628.0  28.469075
Goa                  48372.0    73074.0    78524.0  27.410196
Karnataka          1007894.0  1404447.0  1581988.0  25.283582
Delhi               401540.0   580548.0   606348.0  22.884347
Rajasthan           880985.0  1126130.0  1300476.0  21.497380
Gujarat            1094872.0  1439692.0  1590987.0  20.545677
Assam               379450.0   476195.0   547626.0  20.133672
Mizoram              19439.0    24446.0    27422.0  18.771599
Arunachal Pradesh    19929.0    23726.0    27892.0  18.303359
Haryana             528591.0   642148.0   732029.0  17.680434


In [19]:
## Q8. Peak & low season months for EV sales (2022 – 2024)


ev_all = states.groupby('date')['electric_vehicles_sold'].sum().reset_index()
ev_all['month'] = ev_all.date.dt.month
ev_all['month_name'] = ev_all.date.dt.strftime('%b')

month_avg = (ev_all.groupby('month_name')['electric_vehicles_sold']
             .mean().sort_values(ascending=False))
print("Peak months:\n", month_avg.head(3))
print("Low months:\n",  month_avg.tail(3))

Peak months:
 month_name
Mar    97195.666667
Nov    68398.666667
Feb    66016.333333
Name: electric_vehicles_sold, dtype: float64
Low months:
 month_name
Apr    44885.666667
Jul    42475.333333
Jun    35569.666667
Name: electric_vehicles_sold, dtype: float64


In [20]:
## Q9. Projected EV sales for top 10 states by penetration in 2030

# 1. Identify top 10 states by 2024 penetration
fy24 = states[states.fiscal_year == 2024]
pen24 = (fy24.groupby('state')
         .apply(lambda g: g.electric_vehicles_sold.sum()/g.total_vehicles_sold.sum()*100)
         .sort_values(ascending=False))
top10_states = pen24.head(10).index.tolist()

# 2. Compute CAGR per state on EV sales 2022 -> 2024
ev = (states[states.state.isin(top10_states)]
      .groupby(['state','fiscal_year'])['electric_vehicles_sold']
      .sum().unstack())

ev['CAGR'] = (ev[2024]/ev[2022]) ** (1/2) - 1
ev['Projected_2030'] = ev[2024] * (1 + ev['CAGR']) ** 6   # 2024 → 2030 = 6 yrs

print(ev[['CAGR','Projected_2030']].sort_values('Projected_2030', ascending=False))

fiscal_year       CAGR  Projected_2030
state                                 
Maharashtra   1.018893    1.335115e+07
Kerala        1.328320    1.177940e+07
Karnataka     0.932431    8.383406e+06
Chhattisgarh  1.508917    7.118219e+06
Odisha        1.029421    2.732814e+06
Goa           1.464483    2.419574e+06
Tamil Nadu    0.599531    1.579547e+06
Delhi         0.681001    1.054259e+06
Chandigarh    1.645751    9.868110e+05
Puducherry    1.054436    2.329365e+05


In [21]:
## Q10. Revenue growth rate (2022 vs 2024, 2023 vs 2024)

price = {'2-Wheelers': 85_000, '4-Wheelers': 15_000_000}

rev = (makers.groupby(['vehicle_category','fiscal_year'])
       ['electric_vehicles_sold'].sum().reset_index())
rev['revenue'] = rev.electric_vehicles_sold * rev.vehicle_category.map(price)

piv = rev.pivot(index='vehicle_category', columns='fiscal_year',
                values='revenue')

piv['gr_2022_2024_%'] = (piv[2024]/piv[2022] - 1) * 100
piv['gr_2023_2024_%'] = (piv[2024]/piv[2023] - 1) * 100
print(piv)

fiscal_year               2022          2023           2024  gr_2022_2024_%  \
vehicle_category                                                              
2-Wheelers         21468705000   61871755000    79278820000      269.276209   
4-Wheelers        278655000000  711975000000  1303515000000      367.788125   

fiscal_year       gr_2023_2024_%  
vehicle_category                  
2-Wheelers             28.134106  
4-Wheelers             83.084378  
